In [9]:
"""
Step 1: turn raw CICIDS2018 csv files into one cleaned file that uses the
exact 40 features (and names) of the 2017 training set.

Cleaning mirrors cicids2017_preprocessing.ipynb:
  - drop rows with NaN / inf in the 40 features (NOT filled with 0)
  - drop exact duplicate rows on the 40 features
Extra columns kept for experiments only (never fed to the model):
  Raw Label, Source File
"""
import os
import numpy as np
import pandas as pd


In [10]:

# ---------------- CONFIG: edit paths ----------------
FILES_2018 = [
    r'dataset/IDS2018/BruteForce(FTP,SSH)/BruteForce.csv',
    r'dataset/IDS2018/BruteForce(Web,XSS,SQL)/BruteForce.csv',
    r'dataset/IDS2018/BruteForce(Web,XSS,SQL)/BruteForce2.csv',
    r'dataset/IDS2018\DDoS/DDoS.csv',
    r'dataset/IDS2018/DoS(GoldenEye,Slowloris)/DoS.csv',
    r'dataset/IDS2018/DoS(GoldenEye,Slowloris)/DoS2.csv'
]
OUTPUT_CSV = r'Preprocessing/cicids2018_extra.csv'
# -----------------------------------------------------


In [11]:

FEATURES = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min',
    'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max',
    'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
    'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
    'Flow IAT Min', 'Fwd IAT Std', 'Fwd IAT Min', 'Bwd IAT Total',
    'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length',
    'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length',
    'Packet Length Mean', 'Packet Length Std', 'FIN Flag Count',
    'PSH Flag Count', 'ACK Flag Count', 'Init_Win_bytes_forward',
    'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward',
    'Active Mean', 'Active Max', 'Active Min', 'Idle Mean',
]


In [12]:

# Abbreviated 2018 header style -> feature name. Files that already use the
# 2017-style names pass through unchanged.
ABBREV_TO_FEATURE = {
    'Tot Fwd Pkts': 'Total Fwd Packets', 'Tot Bwd Pkts': 'Total Backward Packets',
    'TotLen Fwd Pkts': 'Total Length of Fwd Packets',
    'Fwd Pkt Len Max': 'Fwd Packet Length Max', 'Fwd Pkt Len Min': 'Fwd Packet Length Min',
    'Fwd Pkt Len Mean': 'Fwd Packet Length Mean', 'Fwd Pkt Len Std': 'Fwd Packet Length Std',
    'Bwd Pkt Len Max': 'Bwd Packet Length Max', 'Bwd Pkt Len Mean': 'Bwd Packet Length Mean',
    'Bwd Pkt Len Std': 'Bwd Packet Length Std', 'Flow Byts/s': 'Flow Bytes/s',
    'Flow Pkts/s': 'Flow Packets/s', 'Bwd IAT Tot': 'Bwd IAT Total',
    'Fwd Header Len': 'Fwd Header Length', 'Bwd Header Len': 'Bwd Header Length',
    'Bwd Pkts/s': 'Bwd Packets/s', 'Pkt Len Min': 'Min Packet Length',
    'Pkt Len Max': 'Max Packet Length', 'Pkt Len Mean': 'Packet Length Mean',
    'Pkt Len Std': 'Packet Length Std', 'FIN Flag Cnt': 'FIN Flag Count',
    'PSH Flag Cnt': 'PSH Flag Count', 'ACK Flag Cnt': 'ACK Flag Count',
    'Init Fwd Win Byts': 'Init_Win_bytes_forward',
    'Init Bwd Win Byts': 'Init_Win_bytes_backward',
    'Fwd Act Data Pkts': 'act_data_pkt_fwd', 'Fwd Seg Size Min': 'min_seg_size_forward',
}


In [13]:

# Keys are lower-cased raw 2018 labels. Anything not listed is dropped and reported.
LABEL_MAP = {
    'benign': 'Normal Traffic',
    'ftp-bruteforce': 'Brute Force',
    'ssh-bruteforce': 'Brute Force',
    'dos attacks-goldeneye': 'DoS',
    'dos attacks-slowloris': 'DoS',
    'dos attacks-hulk': 'DoS',
    'dos attacks-slowhttptest': 'DoS',
    'ddos attacks-loic-http': 'DDoS',
    'ddos attack-loic-udp': 'DDoS',
    'ddos attack-hoic': 'DDoS',
}

NEEDED = set(FEATURES) | set(ABBREV_TO_FEATURE) | {'Label'}



In [17]:

def read_one(path):
    df = pd.read_csv(path, usecols=lambda c: c.strip() in NEEDED, low_memory=False)
    df.columns = df.columns.str.strip()
    df = df.rename(columns=ABBREV_TO_FEATURE)

    missing = [c for c in FEATURES if c not in df.columns]
    if missing:
        raise ValueError(f'{path}: missing features {missing}')

    # 2018 csvs contain repeated header rows inside the data
    df = df[df['Label'].astype(str).str.strip() != 'Label'].copy()

    df['Raw Label'] = df['Label'].astype(str).str.strip()
    df['Source File'] = os.path.basename(os.path.dirname(os.path.abspath(path))) + '/' + os.path.basename(path)
    df['Attack Type'] = df['Raw Label'].str.lower().map(LABEL_MAP)

    dropped = df.loc[df['Attack Type'].isna(), 'Raw Label'].value_counts()
    if len(dropped):
        print(f'  dropped (no class mapping):\n{dropped.to_string()}')
    df = df[df['Attack Type'].notna()]

    df[FEATURES] = df[FEATURES].apply(pd.to_numeric, errors='coerce')
    df[FEATURES] = df[FEATURES].replace([np.inf, -np.inf], np.nan)
    n0 = len(df)
    df = df.dropna(subset=FEATURES)
    print(f'  dropped {n0 - len(df):,} rows with NaN/inf')
    return df[FEATURES + ['Attack Type', 'Raw Label', 'Source File']]



In [18]:

parts = []
for p in FILES_2018:
    print(f'Reading {p}')
    parts.append(read_one(p))

df = pd.concat(parts, ignore_index=True)
_h = pd.util.hash_pandas_object(df[FEATURES], index=False)
_pairs = pd.DataFrame({'h': _h, 'a': df['Attack Type']}).drop_duplicates()
df['_dup'] = _h.duplicated()
df['_conflict'] = _h.isin(set(_pairs['h'][_pairs['h'].duplicated(keep=False)]))
print(df.groupby('Raw Label').agg(rows=('_dup', 'size'), dup_rows=('_dup', 'sum'),
                                  in_label_conflict=('_conflict', 'sum')).to_string())
print(f"Dropping {int(df['_conflict'].sum()):,} rows whose feature vector appears under more than one class")
df = df[~df['_conflict']].drop(columns=['_dup', '_conflict'])
print(f'\nBefore dedup: {len(df):,}')
df = df.drop_duplicates(subset=FEATURES).reset_index(drop=True)
print(f'After dedup:  {len(df):,}')

print('\nRows per Attack Type / Raw Label / Source File:')
print(df.groupby(['Attack Type', 'Raw Label', 'Source File']).size().to_string())

os.makedirs(os.path.dirname(OUTPUT_CSV) or '.', exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved: {OUTPUT_CSV}  shape={df.shape}')

Reading dataset/IDS2018/BruteForce(FTP,SSH)/BruteForce.csv
  dropped 3,824 rows with NaN/inf
Reading dataset/IDS2018/BruteForce(Web,XSS,SQL)/BruteForce.csv
  dropped (no class mapping):
Raw Label
Brute Force -Web    249
Brute Force -XSS     79
SQL Injection        34
  dropped 5,610 rows with NaN/inf
Reading dataset/IDS2018/BruteForce(Web,XSS,SQL)/BruteForce2.csv
  dropped (no class mapping):
Raw Label
Brute Force -Web    362
Brute Force -XSS    151
SQL Injection        53
  dropped 5,708 rows with NaN/inf
Reading dataset/IDS2018\DDoS/DDoS.csv
  dropped 0 rows with NaN/inf
Reading dataset/IDS2018/DoS(GoldenEye,Slowloris)/DoS.csv
  dropped 8,027 rows with NaN/inf
Reading dataset/IDS2018/DoS(GoldenEye,Slowloris)/DoS2.csv
  dropped 0 rows with NaN/inf
                             rows  dup_rows  in_label_conflict
Raw Label                                                     
Benign                    4544367   1035914               5884
DDOS attack-HOIC           686012    487151         